In [1]:
import polars as pl

# Load the unified tract-quarter feature dataset
df = pl.read_parquet("../../data/processed/feature_engineering/tract_quarter_features.parquet")

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD

# 1. Start from full EDA dataframe
# df: Polars DataFrame

# 2. Drop identifiers, target, and QC fields

drop_cols = [
    "tract",                 # identifier
    "reliability_index",     # target
]


# Explicitly collect QC columns by range:
lcdv2_qc_cols = df.columns[
    df.columns.index("weighted_dry_bulb_temp_mean_was_null"):
    df.columns.index("lcdv2_sanitization_level") + 1
]

# ACS/TIGER QC block: population_total_was_null → tract_sanitization_level
acs_tiger_qc_cols = df.columns[
    df.columns.index("population_total_was_null"):
    df.columns.index("tract_sanitization_level") + 1
]

drop_cols.extend(lcdv2_qc_cols)
drop_cols.extend(acs_tiger_qc_cols)

df_features = df.drop(drop_cols)


In [3]:
# quarter: Q1, Q2, Q3, Q4 → 1, 2, 3, 4

df_features = df_features.with_columns(
    pl.when(pl.col("quarter") == "Q1").then(1)
     .when(pl.col("quarter") == "Q2").then(2)
     .when(pl.col("quarter") == "Q3").then(3)
     .when(pl.col("quarter") == "Q4").then(4)
     .otherwise(None)
     .alias("quarter")
)


In [4]:
# Temporal features (already in df_features): year, quarter

# LCDv2 feature block: weighted_dry_bulb_temp_mean → severe_weather_clusters
lcdv2_feature_cols = df_features.columns[
    df_features.columns.index("weighted_dry_bulb_temp_mean"):
    df_features.columns.index("severe_weather_clusters") + 1
]

# ACS/TIGER feature block: population_total → pct_mobile_home
acs_tiger_feature_cols = df_features.columns[
    df_features.columns.index("population_total"):
    df_features.columns.index("pct_mobile_home") + 1
]

# Final feature column list (ordered)
feature_cols = (
    ["year", "quarter"] +
    lcdv2_feature_cols +
    acs_tiger_feature_cols
)

df_matrix = df_features.select(feature_cols)


In [5]:
df_matrix = df_matrix.with_columns([
    pl.col(c).fill_null(0).alias(c)
    for c in df_matrix.columns
])


Performing SVD on feature matrix

In [6]:
X = df_matrix.to_numpy()  # shape: (n_samples, n_features)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [7]:
n_components = 30

svd = TruncatedSVD(n_components=n_components, random_state=42)
X_svd = svd.fit_transform(X_scaled)

components = svd.components_  # shape: (n_components, n_features)
feature_names = df_matrix.columns


In [8]:
import pandas as pd

# Absolute loadings for interpretability
abs_components = np.abs(components)

# Build a DataFrame of loadings
loadings_df = pd.DataFrame(
    abs_components,
    columns=feature_names
)

# For each component, show top 15 features by loading
for i in range(n_components):
    print(f"\nComponent {i+1}")
    print(
        loadings_df.iloc[i]
        .sort_values(ascending=False)
        .head(15)
    )



Component 1
weighted_dew_point_temp_mean      0.105156
weighted_dew_point_temp_p25       0.104769
weighted_dew_point_temp_p10       0.104734
weighted_dew_point_temp_median    0.104649
weighted_wet_bulb_temp_median     0.104545
weighted_wet_bulb_temp_mean       0.104502
weighted_dry_bulb_temp_mean       0.104426
weighted_dry_bulb_temp_median     0.104279
weighted_dew_point_temp_p75       0.104239
weighted_dry_bulb_temp_p25        0.104154
weighted_dry_bulb_temp_p10        0.103860
weighted_wet_bulb_temp_p75        0.103212
weighted_dry_bulb_temp_p75        0.102680
weighted_wet_bulb_temp_p10        0.102525
weighted_wet_bulb_temp_p25        0.101293
Name: 0, dtype: float64

Component 2
weighted_visibility_mean               0.138187
weighted_barometric_pressure_mean      0.123020
weighted_visibility_median             0.121518
weighted_visibility_p25                0.117792
weighted_station_pressure_p75          0.115113
weighted_station_pressure_p90          0.114306
weighted_wind_gus


SVD Cluster Map

Cluster     Description	                    Components
A           LCDv2 Temperature & Moisture	1, 3, 5
B	        LCDv2 Visibility & Pressure	    2, 6, 7
C	        LCDv2 Wind & Gust	            8, 12, 19, 20, 21, 22
D	        LCDv2 Precipitation	            3, 8, 14, 20, 21, 22, 24
E	        LCDv2 Engineered Features	    6, 8, 15, 20, 24
F	        ACS Universe Counts	            4
G	        ACS Education/Income/Poverty	9
H	        TIGER Geometry	                10
I	        ACS Language	                16, 17, 23
J	        ACS Race/Ethnicity	            18, 25, 26, 27, 28, 29, 30
K	        ACS Housing	                    16, 17, 18

Feature Variance and Intra-Cluster Correlation Filtering

In [9]:
# X_scaled: standardized NumPy matrix from Stage 4.1
# df_matrix: Polars DataFrame containing the same columns as X_scaled

feature_names = df_matrix.columns

# Compute variance for each feature
variances = np.var(X_scaled, axis=0)

# Threshold
var_threshold = 1e-4

# Keep features with sufficient variance
keep_var_mask = variances >= var_threshold
keep_var_features = [f for f, keep in zip(feature_names, keep_var_mask) if keep]

print(f"Features kept after variance filtering: {len(keep_var_features)}")
print(f"Features dropped due to low variance: {len(feature_names) - len(keep_var_features)}")

df_var_filtered = df_matrix.select(keep_var_features)


Features kept after variance filtering: 367
Features dropped due to low variance: 2


In [10]:
dropped = [f for f, keep in zip(feature_names, keep_var_mask) if not keep]
print(dropped)


['expected_hours', 'severe_weather_clusters']


In [25]:
clusters = {
    "lcdv2_temperature": [
        # Component 1 + related
        'weighted_dry_bulb_temp_mean', 'weighted_dry_bulb_temp_median', 'weighted_dry_bulb_temp_std', 'weighted_dry_bulb_temp_min', 
        'weighted_dry_bulb_temp_max', 'weighted_dry_bulb_temp_range', 'weighted_dry_bulb_temp_cv', 'weighted_dry_bulb_temp_p10', 
        'weighted_dry_bulb_temp_p25', 'weighted_dry_bulb_temp_p75', 'weighted_dry_bulb_temp_p90', 'weighted_dry_bulb_temp_bucket_-999_0_hours', 
        'weighted_dry_bulb_temp_bucket_-999_0_prop', 'weighted_dry_bulb_temp_bucket_0_10_hours', 'weighted_dry_bulb_temp_bucket_0_10_prop', 
        'weighted_dry_bulb_temp_bucket_10_20_hours', 'weighted_dry_bulb_temp_bucket_10_20_prop', 'weighted_dry_bulb_temp_bucket_20_30_hours', 
        'weighted_dry_bulb_temp_bucket_20_30_prop', 'weighted_dry_bulb_temp_bucket_30_40_hours', 'weighted_dry_bulb_temp_bucket_30_40_prop', 
        'weighted_dry_bulb_temp_bucket_40_50_hours', 'weighted_dry_bulb_temp_bucket_40_50_prop', 'weighted_dry_bulb_temp_bucket_50_60_hours', 
        'weighted_dry_bulb_temp_bucket_50_60_prop', 'weighted_dry_bulb_temp_bucket_60_70_hours', 'weighted_dry_bulb_temp_bucket_60_70_prop', 
        'weighted_dry_bulb_temp_bucket_70_80_hours', 'weighted_dry_bulb_temp_bucket_70_80_prop', 'weighted_dry_bulb_temp_bucket_80_999_hours', 
        'weighted_dry_bulb_temp_bucket_80_999_prop',

        'weighted_wet_bulb_temp_mean', 'weighted_wet_bulb_temp_median', 'weighted_wet_bulb_temp_std', 'weighted_wet_bulb_temp_min', 
        'weighted_wet_bulb_temp_max', 'weighted_wet_bulb_temp_range', 'weighted_wet_bulb_temp_cv', 'weighted_wet_bulb_temp_p10', 
        'weighted_wet_bulb_temp_p25', 'weighted_wet_bulb_temp_p75', 'weighted_wet_bulb_temp_p90', 'weighted_wet_bulb_temp_bucket_-999_0_hours', 
        'weighted_wet_bulb_temp_bucket_-999_0_prop', 'weighted_wet_bulb_temp_bucket_0_10_hours', 'weighted_wet_bulb_temp_bucket_0_10_prop', 
        'weighted_wet_bulb_temp_bucket_10_20_hours', 'weighted_wet_bulb_temp_bucket_10_20_prop', 'weighted_wet_bulb_temp_bucket_20_30_hours', 
        'weighted_wet_bulb_temp_bucket_20_30_prop', 'weighted_wet_bulb_temp_bucket_30_40_hours', 'weighted_wet_bulb_temp_bucket_30_40_prop', 
        'weighted_wet_bulb_temp_bucket_40_50_hours', 'weighted_wet_bulb_temp_bucket_40_50_prop', 'weighted_wet_bulb_temp_bucket_50_60_hours', 
        'weighted_wet_bulb_temp_bucket_50_60_prop', 'weighted_wet_bulb_temp_bucket_60_70_hours', 'weighted_wet_bulb_temp_bucket_60_70_prop', 
        'weighted_wet_bulb_temp_bucket_70_80_hours', 'weighted_wet_bulb_temp_bucket_70_80_prop', 'weighted_wet_bulb_temp_bucket_80_999_hours', 
        'weighted_wet_bulb_temp_bucket_80_999_prop',

        'weighted_dew_point_temp_mean', 'weighted_dew_point_temp_median', 'weighted_dew_point_temp_std', 'weighted_dew_point_temp_min', 
        'weighted_dew_point_temp_max', 'weighted_dew_point_temp_range', 'weighted_dew_point_temp_cv', 'weighted_dew_point_temp_p10', 
        'weighted_dew_point_temp_p25', 'weighted_dew_point_temp_p75', 'weighted_dew_point_temp_p90', 'weighted_dew_point_temp_bucket_-999_0_hours', 
        'weighted_dew_point_temp_bucket_-999_0_prop', 'weighted_dew_point_temp_bucket_0_10_hours', 'weighted_dew_point_temp_bucket_0_10_prop', 
        'weighted_dew_point_temp_bucket_10_20_hours', 'weighted_dew_point_temp_bucket_10_20_prop', 'weighted_dew_point_temp_bucket_20_30_hours', 
        'weighted_dew_point_temp_bucket_20_30_prop', 'weighted_dew_point_temp_bucket_30_40_hours', 'weighted_dew_point_temp_bucket_30_40_prop', 
        'weighted_dew_point_temp_bucket_40_50_hours', 'weighted_dew_point_temp_bucket_40_50_prop', 'weighted_dew_point_temp_bucket_50_60_hours', 
        'weighted_dew_point_temp_bucket_50_60_prop', 'weighted_dew_point_temp_bucket_60_70_hours', 'weighted_dew_point_temp_bucket_60_70_prop', 
        'weighted_dew_point_temp_bucket_70_80_hours', 'weighted_dew_point_temp_bucket_70_80_prop', 'weighted_dew_point_temp_bucket_80_999_hours', 
        'weighted_dew_point_temp_bucket_80_999_prop', 

        'weighted_relative_humidity_mean', 'weighted_relative_humidity_median', 'weighted_relative_humidity_std', 'weighted_relative_humidity_min', 
        'weighted_relative_humidity_max', 'weighted_relative_humidity_range', 'weighted_relative_humidity_cv', 'weighted_relative_humidity_p10', 
        'weighted_relative_humidity_p25', 'weighted_relative_humidity_p75', 'weighted_relative_humidity_p90', 'weighted_relative_humidity_bucket_0_30_hours', 
        'weighted_relative_humidity_bucket_0_30_prop', 'weighted_relative_humidity_bucket_30_50_hours', 'weighted_relative_humidity_bucket_30_50_prop', 
        'weighted_relative_humidity_bucket_50_70_hours', 'weighted_relative_humidity_bucket_50_70_prop', 'weighted_relative_humidity_bucket_70_80_hours', 
        'weighted_relative_humidity_bucket_70_80_prop', 'weighted_relative_humidity_bucket_80_100_hours', 'weighted_relative_humidity_bucket_80_100_prop',
    ],

    "lcdv2_extreme_weather_events": [
        'temp_below_freezing_hours', 'temp_above_90_hours', 'gust_gt_20_hours', 'gust_gt_30_hours', 'gust_gt_40_hours', 'gust_gt_50_hours', 
        'precip_gt_0_01_hours', 'precip_gt_0_10_hours', 'precip_gt_0_25_hours', 'precip_gt_0_50_hours', 'visibility_lt_1_hours', 'low_pressure_hours', 
        'wet_windy_hours', 'cold_windy_hours', 'hot_humid_hours', 'storm_event_hours', 'observed_hours', 'expected_hours', 'coverage_ratio', 
        'wind_dir_resultant_length', 'wind_dir_circular_variance', 'sustained_high_wind_periods', 'severe_weather_clusters',
    ],

    "lcdv2_visibility_pressure": [
        # Components 2, 6, 7
        'weighted_visibility_mean', 'weighted_visibility_median', 'weighted_visibility_std', 'weighted_visibility_min', 
        'weighted_visibility_max', 'weighted_visibility_range', 'weighted_visibility_cv', 'weighted_visibility_p10', 
        'weighted_visibility_p25', 'weighted_visibility_p75', 'weighted_visibility_p90', 'weighted_visibility_bucket_0_1_hours', 
        'weighted_visibility_bucket_0_1_prop', 'weighted_visibility_bucket_1_2_hours', 'weighted_visibility_bucket_1_2_prop', 
        'weighted_visibility_bucket_2_5_hours', 'weighted_visibility_bucket_2_5_prop', 'weighted_visibility_bucket_5_999_hours', 
        'weighted_visibility_bucket_5_999_prop', 

        'weighted_station_pressure_mean', 'weighted_station_pressure_median', 'weighted_station_pressure_std', 'weighted_station_pressure_min', 
        'weighted_station_pressure_max', 'weighted_station_pressure_range', 'weighted_station_pressure_cv', 'weighted_station_pressure_p10', 
        'weighted_station_pressure_p25', 'weighted_station_pressure_p75', 'weighted_station_pressure_p90', 'weighted_station_pressure_bucket_28.0_28.75_hours', 
        'weighted_station_pressure_bucket_28.0_28.75_prop', 'weighted_station_pressure_bucket_28.75_29.25_hours', 'weighted_station_pressure_bucket_28.75_29.25_prop', 
        'weighted_station_pressure_bucket_29.25_29.75_hours', 'weighted_station_pressure_bucket_29.25_29.75_prop', 'weighted_station_pressure_bucket_29.75_31.0_hours', 
        'weighted_station_pressure_bucket_29.75_31.0_prop', 

        'weighted_barometric_pressure_mean', 'weighted_barometric_pressure_median', 'weighted_barometric_pressure_std', 'weighted_barometric_pressure_min', 
        'weighted_barometric_pressure_max', 'weighted_barometric_pressure_range', 'weighted_barometric_pressure_cv', 'weighted_barometric_pressure_p10', 
        'weighted_barometric_pressure_p25', 'weighted_barometric_pressure_p75', 'weighted_barometric_pressure_p90', 'weighted_barometric_pressure_bucket_28.0_28.75_hours', 
        'weighted_barometric_pressure_bucket_28.0_28.75_prop', 'weighted_barometric_pressure_bucket_28.75_29.25_hours', 'weighted_barometric_pressure_bucket_28.75_29.25_prop', 
        'weighted_barometric_pressure_bucket_29.25_29.75_hours', 'weighted_barometric_pressure_bucket_29.25_29.75_prop', 'weighted_barometric_pressure_bucket_29.75_31.0_hours', 
        'weighted_barometric_pressure_bucket_29.75_31.0_prop', 
    ],

    "lcdv2_wind": [
        # Components 8, 12, 19–22
        'weighted_wind_speed_mean', 'weighted_wind_speed_median', 'weighted_wind_speed_std', 'weighted_wind_speed_min', 
        'weighted_wind_speed_max', 'weighted_wind_speed_range', 'weighted_wind_speed_cv', 'weighted_wind_speed_p10', 
        'weighted_wind_speed_p25', 'weighted_wind_speed_p75', 'weighted_wind_speed_p90', 'weighted_wind_speed_bucket_0_5_hours', 
        'weighted_wind_speed_bucket_0_5_prop', 'weighted_wind_speed_bucket_5_10_hours', 'weighted_wind_speed_bucket_5_10_prop', 
        'weighted_wind_speed_bucket_10_15_hours', 'weighted_wind_speed_bucket_10_15_prop', 'weighted_wind_speed_bucket_15_20_hours', 
        'weighted_wind_speed_bucket_15_20_prop', 'weighted_wind_speed_bucket_20_30_hours', 'weighted_wind_speed_bucket_20_30_prop', 
        'weighted_wind_speed_bucket_30_999_hours', 'weighted_wind_speed_bucket_30_999_prop', 

        'weighted_wind_gust_speed_mean', 'weighted_wind_gust_speed_median', 'weighted_wind_gust_speed_std', 'weighted_wind_gust_speed_min', 
        'weighted_wind_gust_speed_max', 'weighted_wind_gust_speed_range', 'weighted_wind_gust_speed_cv', 'weighted_wind_gust_speed_p10', 
        'weighted_wind_gust_speed_p25', 'weighted_wind_gust_speed_p75', 'weighted_wind_gust_speed_p90', 'weighted_wind_gust_speed_bucket_0_20_hours', 
        'weighted_wind_gust_speed_bucket_0_20_prop', 'weighted_wind_gust_speed_bucket_20_30_hours', 'weighted_wind_gust_speed_bucket_20_30_prop', 
        'weighted_wind_gust_speed_bucket_30_40_hours', 'weighted_wind_gust_speed_bucket_30_40_prop', 'weighted_wind_gust_speed_bucket_40_50_hours', 
        'weighted_wind_gust_speed_bucket_40_50_prop', 'weighted_wind_gust_speed_bucket_50_999_hours', 'weighted_wind_gust_speed_bucket_50_999_prop', 
    ],

    "lcdv2_precipitation": [
        # Components 3, 8, 14, 20, 21, 22, 24
        'weighted_precipitation_mean', 'weighted_precipitation_median', 'weighted_precipitation_std', 'weighted_precipitation_min', 
        'weighted_precipitation_max', 'weighted_precipitation_range', 'weighted_precipitation_cv', 'weighted_precipitation_p10', 
        'weighted_precipitation_p25', 'weighted_precipitation_p75', 'weighted_precipitation_p90', 'weighted_precipitation_bucket_-0.0001_0.0001_hours', 
        'weighted_precipitation_bucket_-0.0001_0.0001_prop', 'weighted_precipitation_bucket_0.0001_0.01_hours', 'weighted_precipitation_bucket_0.0001_0.01_prop', 
        'weighted_precipitation_bucket_0.01_0.1_hours', 'weighted_precipitation_bucket_0.01_0.1_prop', 'weighted_precipitation_bucket_0.1_0.25_hours', 
        'weighted_precipitation_bucket_0.1_0.25_prop', 'weighted_precipitation_bucket_0.25_0.5_hours', 'weighted_precipitation_bucket_0.25_0.5_prop', 
        'weighted_precipitation_bucket_0.5_999_hours', 'weighted_precipitation_bucket_0.5_999_prop', 
    ],

    "acs_demographics": [
        'population_total', 'population_male', 'population_female', 'male_65_66', 'male_67_69', 'male_70_74', 'male_75_79', 'male_80_84', 
        'male_85_plus', 'female_65_66', 'female_67_69', 'female_70_74', 'female_75_79', 'female_80_84', 'female_85_plus', 'male_under_5', 
        'female_under_5', 'median_age', 'population_65_plus', 'population_under_5', 'pct_65_plus', 'pct_under_5',  'pct_male', 'pct_female', 
        'sex_ratio',
    ],

    "acs_socioeconomic": [
        # Component 9
        'edu_high_school', 'edu_bachelors', 'edu_masters', 'edu_professional', 'edu_doctorate', 'pct_high_school', 'pct_bachelors', 
        'pct_grad_degree', 'edu_total',

        'poverty_universe', 'poverty_below', 'median_household_income', 'labor_force', 'labor_force_civilian', 'unemployment_civilian', 
        'pct_poverty', 'pct_unemployment',

        'disability_under_18', 'disability_18_64', 'disability_65_plus', 'disability_total',  'pct_disability',

        'housing_structure_universe', 'housing_1_unit', 'housing_2_unit', 'housing_3_4_unit', 'housing_5_9_unit', 'housing_10_19_unit', 
        'housing_20_49_unit', 'housing_50_plus_unit', 'housing_mobile_home', 'pct_housing_occupied', 'pct_housing_vacant', 'housing_units', 
        'housing_occupied', 'housing_vacant', 'median_home_value', 'median_gross_rent', 'pct_single_family', 'pct_small_multi_family', 
        'pct_large_multi_family', 'pct_mobile_home'

        'vehicle_universe', 'vehicle_none', 'pct_vehicle_none',
    ],

    "acs_language": [
        # Components 16, 17, 23
        'language_universe', 'limited_english_5_17', 'limited_english_18_64', 'limited_english_65_plus', 'no_english_5_17', 
        'no_english_18_64', 'no_english_65_plus', 'limited_english_total', 'no_english_total', 'pct_limited_english', 'pct_no_english',
    ],

    "acs_race": [
        # Components 18, 25–30
        'race_universe', 'race_white', 'race_black', 'race_american_indian', 'race_asian', 'race_pacific_islander', 'race_hispanic', 
        'race_other',  'pct_white', 'pct_black', 'pct_asian', 'pct_hispanic', 'pct_american_indian', 'pct_pacific_islander', 'pct_race_other', 
    ],

    "tiger_geometry": [
        # Component 10
        'elevation_m', 'centroid_lat', 'centroid_lon', 'bbox_minx', 'bbox_miny', 'bbox_maxx', 'bbox_maxy', 'tract_area', 
        'tract_perimeter', 'tract_compactness',
    ]
}


In [39]:
corr_threshold = 0.90
df_corr_filtered = df_var_filtered.clone()

for cluster_name, cluster_features in clusters.items():

    # Keep only features that survived variance filtering
    cluster_features = [f for f in cluster_features if f in df_corr_filtered.columns]

    if len(cluster_features) < 2:
        continue

    # Compute correlation matrix
    cluster_df = df_corr_filtered.select(cluster_features)
    corr_matrix = cluster_df.to_pandas().corr().abs()

    # Track features to drop
    to_drop = set()

    # Identify correlated pairs
    for i, f1 in enumerate(cluster_features):
        for j, f2 in enumerate(cluster_features):
            if j <= i:
                continue
            if corr_matrix.iloc[i, j] > corr_threshold:
                # Drop the second feature in the pair
                to_drop.add(f2)

    # Apply drops
    df_corr_filtered = df_corr_filtered.drop(list(to_drop))

    print(f"Cluster {cluster_name}: dropped {len(to_drop)} features")


Cluster lcdv2_temperature: dropped 82 features
Cluster lcdv2_extreme_weather_events: dropped 2 features
Cluster lcdv2_visibility_pressure: dropped 26 features
Cluster lcdv2_wind: dropped 23 features
Cluster lcdv2_precipitation: dropped 12 features
Cluster acs_demographics: dropped 4 features
Cluster acs_socioeconomic: dropped 5 features
Cluster acs_language: dropped 2 features
Cluster acs_race: dropped 4 features
Cluster tiger_geometry: dropped 5 features


In [40]:
final_filtered_features = df_corr_filtered.columns
print(f"Final feature count after filtering: {len(final_filtered_features)}")


Final feature count after filtering: 202


Model-based Importance Feature Analysis for Prediction

In [28]:
from sklearn.model_selection import train_test_split

# X: filtered feature matrix
X = df_corr_filtered.to_numpy()

# y: target from original df
y = df.select("reliability_index").to_numpy().ravel()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

feature_names = df_corr_filtered.columns

In [29]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

rf_importances = rf.feature_importances_


In [30]:
rf_imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_importances
}).sort_values("importance", ascending=False)

rf_imp_df.head(25)

,feature,importance
176,tract_area,0.361307
136,edu_bachelors,0.077201
194,pct_black,0.042502
137,edu_masters,0.029093
187,pct_high_school,0.007970
148,race_white,0.007721
145,median_home_value,0.007586
181,edu_total,0.005993
140,poverty_universe,0.005864
162,vehicle_universe,0.005710


In [31]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    n_jobs=1,
    scoring="neg_mean_squared_error"
)

perm_imp_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

perm_imp_df.head(25)


,feature,importance_mean,importance_std
176,tract_area,485.483605,12.842907
136,edu_bachelors,85.385728,4.793276
137,edu_masters,25.120545,1.954594
194,pct_black,23.305304,2.241652
145,median_home_value,5.716170,1.094373
200,pct_large_multi_family,3.444527,0.548546
170,housing_20_49_unit,2.607773,0.611565
119,male_70_74,2.092828,0.225618
187,pct_high_school,2.077220,0.519522
123,female_65_66,1.906201,0.478496


In [32]:
combined = rf_imp_df.merge(
    perm_imp_df,
    on="feature",
    how="inner"
).sort_values("importance_mean", ascending=False)

combined.head(25)


,feature,importance,importance_mean,importance_std
0,tract_area,0.361307,485.483605,12.842907
1,edu_bachelors,0.077201,85.385728,4.793276
3,edu_masters,0.029093,25.120545,1.954594
2,pct_black,0.042502,23.305304,2.241652
6,median_home_value,0.007586,5.716170,1.094373
18,pct_large_multi_family,0.004600,3.444527,0.548546
55,housing_20_49_unit,0.003211,2.607773,0.611565
20,male_70_74,0.004271,2.092828,0.225618
4,pct_high_school,0.007970,2.077220,0.519522
19,female_65_66,0.004335,1.906201,0.478496


Cluster-Aware Feature Pruning

In [41]:
# Build lookup table for permutation importance
importance_map = {
    row["feature"]: row["importance_mean"]
    for _, row in combined.iterrows()
}

N = 10          # keep top 10 features per cluster
threshold = 0.5 # keep any feature with importance_mean >= 0.5

final_features = []

for cluster_name, cluster_features in clusters.items():

    # Only consider features that survived filtering and have importance scores
    cluster_feats = [
        f for f in cluster_features
        if f in df_corr_filtered.columns and f in importance_map
    ]

    if len(cluster_feats) == 0:
        continue

    # Sort by importance within cluster
    sorted_feats = sorted(
        cluster_feats,
        key=lambda f: importance_map[f],
        reverse=True
    )

    # Keep top N
    keep_topN = sorted_feats[:N]

    # Keep any feature above threshold
    keep_threshold = [
        f for f in cluster_feats
        if importance_map[f] >= threshold
    ]

    # Combine and deduplicate
    keep = set(keep_topN + keep_threshold)

    print(f"Cluster {cluster_name}: keeping {len(keep)} features")

    final_features.extend(keep)


Cluster lcdv2_temperature: keeping 10 features
Cluster lcdv2_extreme_weather_events: keeping 10 features
Cluster lcdv2_visibility_pressure: keeping 10 features
Cluster lcdv2_wind: keeping 10 features
Cluster lcdv2_precipitation: keeping 10 features
Cluster acs_demographics: keeping 10 features
Cluster acs_socioeconomic: keeping 27 features
Cluster acs_language: keeping 9 features
Cluster acs_race: keeping 10 features
Cluster tiger_geometry: keeping 5 features


In [42]:
# Convert pruned feature set to a DataFrame for correlation analysis
df_pruned = df_corr_filtered.select(final_features)

# Compute correlation matrix (absolute value)
corr_matrix = df_pruned.to_pandas().corr().abs()

# Track features to drop
to_drop = set()

# Threshold for cross-cluster correlation
global_corr_threshold = 0.90

# Loop through upper triangle of correlation matrix
for i, f1 in enumerate(final_features):
    if f1 in to_drop:
        continue

    for j in range(i + 1, len(final_features)):
        f2 = final_features[j]

        if f2 in to_drop:
            continue

        corr_value = corr_matrix.iloc[i, j]

        if corr_value > global_corr_threshold:
            # Compare importance scores
            imp1 = importance_map.get(f1, 0)
            imp2 = importance_map.get(f2, 0)

            # Drop the lower-importance feature
            if imp1 >= imp2:
                to_drop.add(f2)
            else:
                to_drop.add(f1)

# Apply global correlation filtering
final_features = [f for f in final_features if f not in to_drop]

print(f"\nCross-cluster correlation filtering removed {len(to_drop)} features.")
print(f"Remaining features after global correlation filtering: {len(final_features)}\n")



Cross-cluster correlation filtering removed 4 features.
Remaining features after global correlation filtering: 107



In [43]:
# temporal features are structurally critical for modeling
# Adding the features to the final feature data set

temporal_features = ["year", "quarter"]

for f in temporal_features:
    if f in df_corr_filtered.columns and f not in final_features:
        final_features.append(f)


In [44]:
# remove dups and sort
final_features = sorted(set(final_features))

# count of and display of final features
print(f"\nTotal retained features: {len(final_features)}\n")
for f in final_features:
    print(f)


Total retained features: 109

centroid_lat
centroid_lon
cold_windy_hours
disability_18_64
disability_total
edu_bachelors
edu_doctorate
edu_high_school
edu_masters
edu_professional
edu_total
elevation_m
female_65_66
female_67_69
female_75_79
gust_gt_30_hours
hot_humid_hours
housing_10_19_unit
housing_1_unit
housing_20_49_unit
housing_3_4_unit
housing_5_9_unit
housing_structure_universe
limited_english_18_64
limited_english_5_17
limited_english_65_plus
low_pressure_hours
male_65_66
male_70_74
male_80_84
median_age
median_gross_rent
median_home_value
median_household_income
no_english_18_64
no_english_5_17
no_english_65_plus
no_english_total
pct_65_plus
pct_asian
pct_bachelors
pct_black
pct_disability
pct_grad_degree
pct_high_school
pct_housing_occupied
pct_large_multi_family
pct_limited_english
pct_poverty
pct_single_family
pct_small_multi_family
pct_under_5
pct_unemployment
pct_white
population_65_plus
precip_gt_0_50_hours
quarter
race_american_indian
race_asian
race_black
race_hispani

In [48]:
# build final dataframe and save output for modeling
df_final = df_corr_filtered.select(final_features)

# Add target column from original df
df_final = df_final.with_columns(
    df.select("reliability_index").to_series()
)

output_path = "../../data/processed/modeling/final_feature_set.parquet"
df_final.write_parquet(output_path)

print(f"\nFinal feature set saved to: {output_path}")



Final feature set saved to: ../../data/processed/modeling/final_feature_set.parquet
